In [1]:
# Set up the file path
import os
os.chdir('..')

In [2]:
# Import packages
from RL4CRN.policies.parameter_generator_from_distribution import ParameterGeneratorFromDistribution
from RL4CRN.iocrns.reaction_library import construct_hill_production_library
import torch

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# Create a library
library = construct_hill_production_library(species_labels=['X_1', 'X_2', 'X_3'], max_product_order=1, max_num_regulators=2)
M = len(library.reactions) # Number of possible reactions

In [5]:
# Construct the continuous parameter masks
continuous_parameter_mask = library.get_parameter_mask(mode="continuous")
continuous_parameter_mask = torch.tensor(continuous_parameter_mask, dtype=torch.float32).to(device) if continuous_parameter_mask is not None else None # Shape: (M, max_num_continuous_parameters)
D = continuous_parameter_mask.shape[1] if continuous_parameter_mask is not None else 0 # Number of continuous parameters per reaction

In [6]:
# Create an instance of ParameterGeneratorFromDistribution
d = 1000
h = 128
n = 3
continuous_parameter_generator = ParameterGeneratorFromDistribution(
    distribution={"type": 'lognormal', "dim": D}, 
    backbone_attributes={"input_size": d + M, 
                         "hidden_size": h, 
                         "num_layers": n
                        }, 
    device=device
    ).to(device=device)

In [7]:
# Generate a batch of reaction indices
N = 4
samples_reaction_idx = torch.randint(low=0, high=M, size=(N,), device=device)

# Map the masks to the sampled reaction indices
continuous_parameter_mask_subset = continuous_parameter_mask[samples_reaction_idx] if continuous_parameter_mask is not None else None

In [8]:
# Construct a batch of input data to test the forward pass
N = 4
x = torch.randn((N, d + M), device=device) 
samples, log_probs, entropies = continuous_parameter_generator(x, continuous_parameter_mask_subset) 
print(continuous_parameter_mask_subset)
print(samples)

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]], device='cuda:0')
tensor([[1.2967, 0.1927, 0.3867, 2.0766],
        [0.1464, 0.3490, 1.0560, 1.7046],
        [0.0862, 0.8869, 0.6858, 0.3920],
        [0.5715, 0.2154, 0.3966, 0.8399]], device='cuda:0')


In [9]:
from RL4CRN.distributions.lognormal import MultivariateLogNormal
mask = torch.tensor([1.0, 1.0, 1.0, 0.0])
m = torch.tensor([1.0, 2.0, 3.0, 4.0])
S = 0.001*torch.eye(4)
outer_mm = m.unsqueeze(-1) * m.unsqueeze(-2)                            # shape: (N, D, D)
Sigma = torch.log1p(S / outer_mm)                                       # shape: (N, D, D)
mu = torch.log(m) - 0.5 * torch.diagonal(Sigma, dim1=-1, dim2=-2)

mu = mu.masked_fill(~mask.bool(), float('-inf'))
mask_soft = torch.where(mask == 0, torch.tensor(1e-8, dtype=mask.dtype, device=mask.device), mask)
Sigma = Sigma * mask_soft.unsqueeze(-1) * mask_soft.unsqueeze(-2)

dist = MultivariateLogNormal(loc=mu, covariance_matrix=Sigma)
samples = dist.sample((10,))

In [10]:
print(samples)
print(mask)

tensor([[0.9819, 2.0159, 3.0178, 0.0000],
        [0.9452, 2.0519, 3.0253, 0.0000],
        [0.9870, 2.0196, 3.0146, 0.0000],
        [0.9816, 2.0161, 2.9760, 0.0000],
        [1.0321, 2.0221, 2.9756, 0.0000],
        [1.0013, 2.0161, 2.9857, 0.0000],
        [1.0686, 1.9458, 3.0364, 0.0000],
        [1.0417, 1.9856, 2.9607, 0.0000],
        [1.0126, 2.0195, 3.0271, 0.0000],
        [0.9940, 1.9985, 3.0499, 0.0000]])
tensor([1., 1., 1., 0.])
